In [ ]:
# CNN para Classificação de Pedestres com PyTorch no Google Colab

# --- Etapa 1: Carregamento e Preparação dos Dados ---
import os
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Carregar imagens
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

imagens = []
saidas = []

for arquivo in os.scandir('Background'):
    imagem = cv2.imread(arquivo.path)
    imagem = cv2.resize(imagem, (224, 224))
    imagens.append(imagem)
    saidas.append(0)  # Não-Pedestre

for arquivo in os.scandir('Pedestrians'):
    imagem = cv2.imread(arquivo.path)
    imagem = cv2.resize(imagem, (224, 224))
    imagens.append(imagem)
    saidas.append(1)  # Pedestre

X = np.array(imagens, dtype=np.float32) / 255.0
y = np.array(saidas)

# Divisão do dataset completo em treino, validação e teste (70%, 15%, 15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

X_train_tensor = torch.tensor(X_train).permute(0, 3, 1, 2)
X_val_tensor = torch.tensor(X_val).permute(0, 3, 1, 2)
X_test_tensor = torch.tensor(X_test).permute(0, 3, 1, 2)

y_train_tensor = torch.tensor(y_train).long()
y_val_tensor = torch.tensor(y_val).long()
y_test_tensor = torch.tensor(y_test).long()

batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [ ]:

# --- Etapa 2: Definir Arquitetura CNN ---
import torch.nn as nn
import torch.nn.functional as F

class PedestrianCNN(nn.Module):
    def __init__(self):
        super(PedestrianCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:

# --- Etapa 3: Treinamento com Validação ---
model = PedestrianCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_losses.append(running_loss / len(train_loader))

    model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_losses.append(val_loss / len(val_loader))
    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, Val Accuracy: {accuracy:.2f}%")


In [ ]:

# --- Etapa 4: Avaliação no Teste ---
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print(f"Acurácia: {accuracy:.2f}")
print(f"Precisão: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")
print("Matriz de Confusão:")
print(cm)


In [ ]:

# --- Etapa 5: Salvar o Modelo ---
torch.save(model.state_dict(), "modelo_cnn_pedestre.pth")
print("Modelo salvo como 'modelo_cnn_pedestre.pth'")


In [ ]:

# --- Etapa Final: Visualização da saída esperada de todas imagens do Dataset ---
import matplotlib.pyplot as plt

model.eval()
total_exibidas = 0

for inputs, labels in DataLoader(TensorDataset(torch.tensor(X).permute(0, 3, 1, 2), torch.tensor(y)), batch_size=1):
    inputs = inputs.to(device)
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)

    imagem = inputs[0].cpu().permute(1, 2, 0).numpy()
    saida_esperada = labels.item()

    plt.figure(figsize=(4, 4))
    plt.imshow(imagem)
    plt.title(f"Saida esperada: {saida_esperada}")
    plt.axis('off')
    plt.show()
    print("\n")
    total_exibidas += 1
